# Notebook 03 — Bidirectional RNN with Pre-trained Embeddings
**Project:** Sentiment Analysis Through the Ages  
**Tier:** 2 of 3 — BiLSTM + GloVe / Word2Vec  
**Dataset:** SST-2 (binary) and SST-5 (fine-grained)

---

## What This Notebook Does

We replace the N-gram bag-of-words representation with a **Bidirectional LSTM** that reads token sequences left-to-right *and* right-to-left simultaneously, using **pre-trained GloVe or Word2Vec embeddings** as input.

### Core concepts demonstrated:
| Concept | Where |
|---|---|
| Tokenization (spaCy) | Section 2 |
| GloVe embeddings | Section 3 |
| Word2Vec embeddings | Section 3 |
| Recurrent Neural Networks (LSTM) | Section 4 |
| Bidirectional RNN | Section 4 |
| First-order optimisation (Adam) | Section 5 |
| Gradient clipping | Section 5 |
| Gradient flow / vanishing gradient | Section 7 |
| Embedding ablation (GloVe vs Word2Vec) | Section 8 |
| Evaluation + learning curves | Section 6 |

## 0. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q datasets torch spacy gensim
    !python -m spacy download en_core_web_sm -q

    # Download GloVe (100d, ~350MB)
    import os
    if not os.path.exists('data/glove.6B.100d.txt'):
        !mkdir -p data
        !wget -q https://nlp.stanford.edu/data/glove.6B.zip -O data/glove.zip
        !unzip -q data/glove.zip -d data/
        !rm data/glove.zip
        print('GloVe downloaded.')

import os, re, time, sys
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)

# Add project root to path so we can import from models/
sys.path.insert(0, '..')
from models.birnn import (
    clean_text, load_sst, spacy_tokenize,
    Vocabulary, load_glove, load_word2vec, make_random_embeddings,
    SSTDataset, BiLSTMClassifier,
    train_epoch, evaluate_epoch, plot_gradient_flow,
    DEVICE,
)

os.makedirs('results', exist_ok=True)
print(f'Device: {DEVICE}')
print('Setup complete.')

---
## 1. Load and Inspect Data

In [ ]:
# ── SST-2 ─────────────────────────────────────────────────────────────────
# stanfordnlp/sst2: text field = 'sentence', label field = 'label' (0/1)
train2_texts, train2_labels, val2_texts, val2_labels, label_names_2 = load_sst(2)

# ── SST-5 ─────────────────────────────────────────────────────────────────
# SetFit/sst5: text field = 'text', label field = 'label' (0-4)
train5_texts, train5_labels, val5_texts, val5_labels, label_names_5 = load_sst(5)

# Show sample after cleaning
print('\nSST-2 sample (cleaned):')
for i in range(3):
    print(f'  [{label_names_2[train2_labels[i]]}] "{train2_texts[i]}"')

print('\nSST-5 sample (cleaned):')
for i in range(3):
    print(f'  [{label_names_5[train5_labels[i]]}] "{train5_texts[i]}"')

---
## 2. Tokenisation (spaCy)

Unlike Tier 1 (regex whitespace split), we now use spaCy's tokeniser. It correctly handles:
- Punctuation attached to words (`"film,"` → `["film", ","]`)
- Hyphenated words
- Apostrophes in contractions

This is an explicit step forward in tokenisation quality between Tiers 1 and 2.

In [ ]:
print('Tokenising SST-2 training set with spaCy...')
train2_tokens = spacy_tokenize(train2_texts)
val2_tokens   = spacy_tokenize(val2_texts)

print('Tokenising SST-5 training set with spaCy...')
train5_tokens = spacy_tokenize(train5_texts)
val5_tokens   = spacy_tokenize(val5_texts)

# Compare tokenisers on an example
ex = train2_texts[36]  # a sentence with contractions
import re
tier1_toks = re.findall(r"[a-z']+", ex.lower())
tier2_toks = train2_tokens[36]
print(f'\nExample: "{ex}"')
print(f'  Tier 1 (regex):  {tier1_toks}')
print(f'  Tier 2 (spaCy):  {tier2_toks}')

---
## 3. Vocabulary and Embeddings

### GloVe vs Word2Vec — conceptual difference

| Property | Word2Vec (Skip-gram) | GloVe |
|---|---|---|
| Training objective | Predict context from centre word | Factorise global co-occurrence matrix |
| Context | Local window (5–10 words) | Corpus-wide statistics |
| Geometry | Linear analogies (king-man+woman≈queen) | Same, but also encodes co-occurrence ratios |
| Typical dim | 300d (Google News) | 50d, 100d, 200d, 300d |
| OOV | Unseen words need random init | Same |

In [ ]:
# Build vocabulary from SST-2 training tokens
vocab2 = Vocabulary(min_freq=2).build(train2_tokens)
vocab5 = Vocabulary(min_freq=2).build(train5_tokens)

print(f'SST-2 vocab: {len(vocab2):,} tokens')
print(f'SST-5 vocab: {len(vocab5):,} tokens')

# Token length statistics
lengths = [len(t) for t in train2_tokens]
print(f'\nSST-2 token lengths — mean: {np.mean(lengths):.1f} | '
      f'median: {np.median(lengths):.0f} | max: {max(lengths)}')
print(f'  Sentences ≤64 tokens: {sum(l<=64 for l in lengths)/len(lengths)*100:.1f}%  '
      f'→ max_len=64 covers virtually all of SST')

In [ ]:
# ── Load GloVe (or fall back to random if file not found) ─────────────────
GLOVE_PATH = 'data/glove.6B.100d.txt'
EMBEDDING_DIM = 100

if Path(GLOVE_PATH).exists():
    glove_emb2 = load_glove(GLOVE_PATH, vocab2, embedding_dim=EMBEDDING_DIM)
    glove_emb5 = load_glove(GLOVE_PATH, vocab5, embedding_dim=EMBEDDING_DIM)
else:
    print(f'GloVe not found at {GLOVE_PATH}.')
    print('Using random embeddings. To use GloVe:')
    print('  wget https://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip -d data/')
    glove_emb2 = make_random_embeddings(vocab2, EMBEDDING_DIM)
    glove_emb5 = make_random_embeddings(vocab5, EMBEDDING_DIM)

In [ ]:
# ── Visualise embedding space (t-SNE on a sample of sentiment words) ───────
if Path(GLOVE_PATH).exists():
    from sklearn.manifold import TSNE

    sentiment_words = [
        'brilliant', 'masterpiece', 'stunning', 'beautiful', 'excellent',
        'awful', 'terrible', 'boring', 'dull', 'waste',
        'not', 'never', 'barely', 'hardly',
        'film', 'movie', 'director', 'actor', 'story',
    ]
    sentiments = (
        ['positive'] * 5 + ['negative'] * 5 +
        ['negation'] * 4 + ['neutral'] * 5
    )
    color_map = {'positive': '#4C72B0', 'negative': '#DD8452',
                 'negation': '#55A868', 'neutral': '#C44E52'}

    indices = [vocab2.token2idx.get(w, vocab2.UNK_IDX) for w in sentiment_words]
    vecs = glove_emb2[indices].numpy()

    tsne = TSNE(n_components=2, random_state=42, perplexity=5)
    coords = tsne.fit_transform(vecs)

    plt.figure(figsize=(9, 6))
    for word, coord, sent in zip(sentiment_words, coords, sentiments):
        plt.scatter(*coord, color=color_map[sent], s=60, zorder=2)
        plt.annotate(word, coord, textcoords='offset points',
                     xytext=(5, 3), fontsize=9)
    for label, color in color_map.items():
        plt.scatter([], [], color=color, label=label, s=60)
    plt.legend(title='Word type')
    plt.title('t-SNE of GloVe Embeddings for Sentiment-Relevant Words')
    plt.tight_layout()
    plt.savefig('results/tsne_glove_embeddings.png', dpi=150)
    plt.show()
    print('Observation: Positive and negative words cluster separately —')
    print('GloVe encodes semantic similarity that the TF-IDF baseline cannot capture.')

---
## 4. BiLSTM Model Architecture

### Why Bidirectional?

Consider the sentence: *"The film wasn't particularly **good**, but it wasn't **terrible** either."*

A forward LSTM at the word *"good"* has already seen *"wasn't"* — so it knows the negation. But a forward LSTM at *"terrible"* hasn't yet seen *"either"* (which weakens the negative). A **backward** LSTM fills this gap.

The BiLSTM concatenates `[forward_hidden, backward_hidden]` at each position, giving each token access to its full left *and* right context.

In [ ]:
# ── Build DataLoaders for SST-2 ────────────────────────────────────────────
MAX_LEN    = 64
BATCH_SIZE = 64

train2_ds = SSTDataset(train2_tokens, train2_labels, vocab2, MAX_LEN)
val2_ds   = SSTDataset(val2_tokens,   val2_labels,   vocab2, MAX_LEN)
train2_loader = DataLoader(train2_ds, batch_size=BATCH_SIZE, shuffle=True)
val2_loader   = DataLoader(val2_ds,   batch_size=BATCH_SIZE)

# Show a batch
x, lengths, y = next(iter(train2_loader))
print(f'Batch shape — input_ids: {x.shape} | lengths: {lengths[:5]} | labels: {y[:5]}')

In [ ]:
# ── Instantiate BiLSTM ─────────────────────────────────────────────────────
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.3

model_sst2 = BiLSTMClassifier(
    vocab_size     = len(vocab2),
    embedding_dim  = EMBEDDING_DIM,
    hidden_dim     = HIDDEN_DIM,
    n_layers       = N_LAYERS,
    n_classes      = 2,
    dropout        = DROPOUT,
    pretrained_emb = glove_emb2,
    freeze_emb     = False,   # allow embedding fine-tuning
).to(DEVICE)

print(f'Trainable parameters: {model_sst2.count_parameters():,}')
print(f'Model architecture:')
print(model_sst2)

---
## 5. Training — Adam Optimiser (First-Order)

**Why Adam here vs L-BFGS in Tier 1?**

L-BFGS approximates the Hessian and works well for **convex, low-dimensional** problems (logistic regression). Neural networks are high-dimensional and non-convex — computing even an approximate Hessian is prohibitively expensive. Adam is a **first-order** method that adapts the learning rate per parameter using only gradient information, making it the standard choice for deep learning.

In [ ]:
# Training configuration
N_EPOCHS  = 10
LR        = 1e-3
PATIENCE  = 3

class_counts  = np.bincount(train2_labels)
class_weights = torch.tensor(
    1.0 / (class_counts / class_counts.sum()), dtype=torch.float
).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Adam: first-order adaptive optimiser
optimizer = torch.optim.Adam(
    model_sst2.parameters(), lr=LR, weight_decay=1e-5
)
# Reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5, verbose=True
)

print(f'Optimiser: Adam (lr={LR}) | Loss: CrossEntropyLoss (weighted)')

In [ ]:
history2 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss2, best_state2, patience_ctr = float('inf'), None, 0

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model_sst2, train2_loader, optimizer, criterion)
    va_loss, va_acc, _, _ = evaluate_epoch(model_sst2, val2_loader, criterion)
    scheduler.step(va_loss)

    history2['train_loss'].append(tr_loss)
    history2['val_loss'].append(va_loss)
    history2['train_acc'].append(tr_acc)
    history2['val_acc'].append(va_acc)

    print(f'Epoch {epoch:2d}/{N_EPOCHS}  '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f}  '
          f'val_loss={va_loss:.4f} acc={va_acc:.4f}  '
          f'({time.time()-t0:.1f}s)')

    if va_loss < best_val_loss2:
        best_val_loss2 = va_loss
        best_state2 = {k: v.cpu().clone() for k, v in model_sst2.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

model_sst2.load_state_dict(best_state2)

---
## 6. Evaluation — SST-2

In [ ]:
_, _, y_pred2, y_true2 = evaluate_epoch(model_sst2, val2_loader, criterion)
acc2 = accuracy_score(y_true2, y_pred2)
f1_2 = f1_score(y_true2, y_pred2, average='macro')

print(f'BiLSTM + GloVe — SST-2')
print(f'  Accuracy  : {acc2:.4f} ({acc2*100:.2f}%)')
print(f'  Macro-F1  : {f1_2:.4f}')
print()
print(classification_report(y_true2, y_pred2, target_names=label_names_2))

In [ ]:
# Confusion matrix
cm2 = confusion_matrix(y_true2, y_pred2)
plt.figure(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_2, yticklabels=label_names_2)
plt.title('Confusion Matrix — BiLSTM + GloVe (SST-2)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('results/confusion_birnn_sst2_glove.png', dpi=150)
plt.show()

In [ ]:
# Learning curves
epochs_ran = len(history2['train_loss'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, epochs_ran+1), history2['train_loss'], 'o-', label='Train', color='#4C72B0')
ax1.plot(range(1, epochs_ran+1), history2['val_loss'],   's--', label='Val',   color='#DD8452')
ax1.set_title('Loss — BiLSTM + GloVe (SST-2)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(range(1, epochs_ran+1), history2['train_acc'], 'o-', label='Train', color='#4C72B0')
ax2.plot(range(1, epochs_ran+1), history2['val_acc'],   's--', label='Val',   color='#DD8452')
ax2.set_title('Accuracy — BiLSTM + GloVe (SST-2)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/learning_curve_birnn_sst2_glove.png', dpi=150)
plt.show()

---
## 7. Gradient Flow Analysis

This plot reveals whether gradients are flowing through all layers of the BiLSTM. Near-zero gradients in early layers indicate the **vanishing gradient problem** — the model's early layers stop learning. This directly motivates the move to BERT (Tier 3), which uses residual connections and attention to avoid this entirely.

In [ ]:
# Trigger one backward pass to populate .grad attributes
model_sst2.train()
sample_x, sample_l, sample_y = next(iter(train2_loader))
loss = criterion(
    model_sst2(sample_x.to(DEVICE), sample_l.to(DEVICE)),
    sample_y.to(DEVICE)
)
loss.backward()

plot_gradient_flow(model_sst2,
                   save_path='results/gradient_flow_birnn_sst2.png')

---
## 8. SST-5 Fine-Grained Classification

In [ ]:
# Build SST-5 DataLoaders
train5_ds = SSTDataset(train5_tokens, train5_labels, vocab5, MAX_LEN)
val5_ds   = SSTDataset(val5_tokens,   val5_labels,   vocab5, MAX_LEN)
train5_loader = DataLoader(train5_ds, batch_size=BATCH_SIZE, shuffle=True)
val5_loader   = DataLoader(val5_ds,   batch_size=BATCH_SIZE)

# New model for SST-5
model_sst5 = BiLSTMClassifier(
    vocab_size=len(vocab5), embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, n_classes=5,
    dropout=DROPOUT, pretrained_emb=glove_emb5, freeze_emb=False,
).to(DEVICE)

class_counts5  = np.bincount(train5_labels)
class_weights5 = torch.tensor(
    1.0 / (class_counts5 / class_counts5.sum()), dtype=torch.float
).to(DEVICE)
criterion5  = nn.CrossEntropyLoss(weight=class_weights5)
optimizer5  = torch.optim.Adam(model_sst5.parameters(), lr=LR, weight_decay=1e-5)
scheduler5  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer5, mode='min', patience=2, factor=0.5
)

history5 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss5, best_state5, patience_ctr5 = float('inf'), None, 0

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model_sst5, train5_loader, optimizer5, criterion5)
    va_loss, va_acc, _, _ = evaluate_epoch(model_sst5, val5_loader, criterion5)
    scheduler5.step(va_loss)
    history5['train_loss'].append(tr_loss)
    history5['val_loss'].append(va_loss)
    history5['train_acc'].append(tr_acc)
    history5['val_acc'].append(va_acc)
    print(f'Epoch {epoch:2d}/{N_EPOCHS}  tr={tr_loss:.4f}/{tr_acc:.4f}  '
          f'val={va_loss:.4f}/{va_acc:.4f}  ({time.time()-t0:.1f}s)')
    if va_loss < best_val_loss5:
        best_val_loss5 = va_loss
        best_state5 = {k: v.cpu().clone() for k, v in model_sst5.state_dict().items()}
        patience_ctr5 = 0
    else:
        patience_ctr5 += 1
        if patience_ctr5 >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.'); break

model_sst5.load_state_dict(best_state5)
_, _, y_pred5, y_true5 = evaluate_epoch(model_sst5, val5_loader, criterion5)
acc5 = accuracy_score(y_true5, y_pred5)
f1_5 = f1_score(y_true5, y_pred5, average='macro')

print(f'\nBiLSTM + GloVe — SST-5')
print(f'  Accuracy : {acc5:.4f} | Macro-F1 : {f1_5:.4f}')
print(classification_report(y_true5, y_pred5, target_names=label_names_5))

In [ ]:
cm5 = confusion_matrix(y_true5, y_pred5)
plt.figure(figsize=(7, 6))
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_5, yticklabels=label_names_5)
plt.title('Confusion Matrix — BiLSTM + GloVe (SST-5)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('results/confusion_birnn_sst5_glove.png', dpi=150)
plt.show()

---
## 9. Embedding Ablation: GloVe vs Word2Vec vs Random

In [ ]:
# Run ablation on SST-2 only (for speed)
# All other hyperparameters held constant — only embedding changes

def quick_train_eval(pretrained_emb, label, vocab, n_classes,
                     train_loader, val_loader, train_labels,
                     n_epochs=5):
    emb_dim = pretrained_emb.shape[1]
    m = BiLSTMClassifier(
        vocab_size=len(vocab), embedding_dim=emb_dim,
        hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS,
        n_classes=n_classes, dropout=DROPOUT,
        pretrained_emb=pretrained_emb, freeze_emb=False,
    ).to(DEVICE)

    cc = np.bincount(train_labels)
    cw = torch.tensor(1.0/(cc/cc.sum()), dtype=torch.float).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    opt  = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=1e-5)

    for ep in range(n_epochs):
        train_epoch(m, train_loader, opt, crit)

    _, _, preds, trues = evaluate_epoch(m, val_loader, crit)
    acc = accuracy_score(trues, preds)
    mf1 = f1_score(trues, preds, average='macro')
    print(f'  {label:>20}: acc={acc:.4f}  macro-f1={mf1:.4f}')
    return {'embedding': label, 'accuracy': round(acc,4), 'macro_f1': round(mf1,4)}

print('Embedding ablation (SST-2, 5 epochs each):')
ablation_rows = []

# GloVe
ablation_rows.append(quick_train_eval(
    glove_emb2, 'GloVe 100d', vocab2, 2,
    train2_loader, val2_loader, train2_labels
))

# Random init (same dim as GloVe)
rand_emb = make_random_embeddings(vocab2, EMBEDDING_DIM)
ablation_rows.append(quick_train_eval(
    rand_emb, 'Random 100d', vocab2, 2,
    train2_loader, val2_loader, train2_labels
))

# Word2Vec (if available)
W2V_PATH = 'data/GoogleNews-vectors-negative300.bin'
if Path(W2V_PATH).exists():
    w2v_emb2 = load_word2vec(W2V_PATH, vocab2, embedding_dim=300)
    ablation_rows.append(quick_train_eval(
        w2v_emb2, 'Word2Vec 300d', vocab2, 2,
        train2_loader, val2_loader, train2_labels
    ))
else:
    print(f'  Word2Vec not found at {W2V_PATH} — skipping.')
    print('  Download: https://code.google.com/archive/p/word2vec/')

ablation_df = pd.DataFrame(ablation_rows)
print('\nAblation summary:')
print(ablation_df.to_string(index=False))

In [ ]:
# Plot ablation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = sns.color_palette('Blues_d', len(ablation_df))
for ax, metric in zip(axes, ['accuracy', 'macro_f1']):
    bars = ax.bar(ablation_df['embedding'], ablation_df[metric], color=colors)
    ax.set_title(f'Embedding Ablation — {metric} (SST-2)')
    ax.set_ylabel(metric)
    ymin = ablation_df[metric].min() - 0.03
    ymax = ablation_df[metric].max() + 0.02
    ax.set_ylim(ymin, ymax)
    for bar, val in zip(bars, ablation_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('results/embedding_ablation.png', dpi=150)
plt.show()

---
## 10. Tier Summary & Comparison with Baseline

In [ ]:
# Load baseline results if available
try:
    baseline_df = pd.read_csv('results/baseline_results.csv', index_col=0)
    baseline_sst2_acc = baseline_df.loc['SST-2', 'accuracy']
    baseline_sst5_acc = baseline_df.loc['SST-5', 'accuracy']
except FileNotFoundError:
    baseline_sst2_acc = 0.840   # literature value
    baseline_sst5_acc = 0.430
    print('Baseline results file not found — using literature values.')

comparison = pd.DataFrame([
    {'Model': 'N-gram Baseline (Tier 1)', 'SST-2 Acc': baseline_sst2_acc, 'SST-5 Acc': baseline_sst5_acc},
    {'Model': 'BiLSTM + GloVe (Tier 2)', 'SST-2 Acc': acc2,              'SST-5 Acc': acc5},
    {'Model': 'BERT (Tier 3 — target)',   'SST-2 Acc': 0.930,             'SST-5 Acc': 0.530},
]).set_index('Model')

print('═' * 55)
print('  TIER 1 vs TIER 2 COMPARISON')
print('═' * 55)
print(comparison.round(4).to_string())

gain_sst2 = acc2 - baseline_sst2_acc
gain_sst5 = acc5 - baseline_sst5_acc
print(f'\nSST-2 gain over baseline : +{gain_sst2:.4f} ({gain_sst2*100:.1f} pp)')
print(f'SST-5 gain over baseline : +{gain_sst5:.4f} ({gain_sst5*100:.1f} pp)')

# Save for final comparison table
comparison.to_csv('results/tier1_vs_tier2.csv')